# 第 6 周练习：亚马逊风格价格预测器

## 练习目标（理念）

用**合成的亚马逊风格商品数据**，对比多条路线预测价格档位（或价格本身）：

- 传统机器学习：TF-IDF + Logistic / Random Forest /（可选）XGBoost
- 小型神经网络：PyTorch MLP
- （可选）通过 OpenRouter 调用云端 LLM 做零样本（zero-shot）分类
- Gradio 小界面：把模型能力做成可点点的 Demo

这对应课程第 6 周「价格预测 / 微调与基线对比」的社区练习版：先把数据与基线跑通，再考虑 LLM。

## 怎么跑（务必从上到下）

**重要：** 按顺序运行。跳过中间格容易 `NameError`（例如没有 `df` / `results`）。

1. **重启内核**：`Kernel` → `Restart Kernel`（清掉旧变量）
2. **单元 1**：导入依赖（Mac 上可能看到 XGBoost / libomp 警告，可忽略）
3. **单元 2**：`Config` 与随机种子
4. **单元 3**：生成合成商品表 `df`
5. **其余单元**：逐格运行，或重启后用 `Kernel` → `Run All`

**快捷方式：** 重启后直接 **Run All**。


In [ ]:
# ========== 导入：数据 / 传统 ML / 深度学习 / OpenRouter / Gradio ==========

# os/json/numpy/pandas/matplotlib：环境、序列化、数值、表格、画图
import os, json, numpy as np, pandas as pd, matplotlib.pyplot as plt
# datetime：给结果文件打时间戳
from datetime import datetime
# load_dotenv：从 .env 读 OPENROUTER_API_KEY 等，避免密钥进笔记本
from dotenv import load_dotenv
# tqdm：进度条（本格主要预留；后面循环可用）
from tqdm import tqdm
# 屏蔽无关告警，输出更干净
import warnings
warnings.filterwarnings('ignore')

# --- 机器学习（scikit-learn） ---
# 划分训练/测试集
from sklearn.model_selection import train_test_split
# 把价格档位字符串编码成整数标签
from sklearn.preprocessing import LabelEncoder
# 线性回归（价格数值）与逻辑回归（档位分类）
from sklearn.linear_model import LinearRegression, LogisticRegression
# 随机森林回归/分类（本练习主要用分类器）
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
# TF-IDF：把标题+描述等文本变成稀疏特征向量
from sklearn.feature_extraction.text import TfidfVectorizer
# 分类准确率、回归平均绝对误差
from sklearn.metrics import accuracy_score, mean_absolute_error
# XGBoost：可选；缺 libomp 时置 None，后面跳过该模型
try:
    import xgboost as xgb
except Exception as e:
    print("⚠️ XGBoost not available (on Mac run: brew install libomp). Skipping XGBoost model.")
    xgb = None

# --- 深度学习（PyTorch） ---
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

# Hugging Face datasets：可选上传到 Hub 时用（别名 HFDataset，避免与 torch Dataset 撞名）
from datasets import Dataset as HFDataset

# OpenAI 兼容客户端：用来打 OpenRouter
from openai import OpenAI
# OpenRouter 的 OpenAI 兼容 base URL（字符串保持原样）
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"

# Gradio：快速搭交互 Demo
import gradio as gr

# 加载 .env 到进程环境
load_dotenv()


In [ ]:
# ========== Config：模型名、API Key、数据规模、价格分箱与随机种子 ==========

# 用类集中放超参，后面一律 config.xxx，避免魔法数字散落
class Config:
    # OpenRouter 上的模型 id（可运行字符串，勿改）
    BASE_MODEL = "openai/gpt-4o-mini"
    # 从环境变量读 Key；没有则后续 LLM / Gradio 分支会跳过
    OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
    # 合成商品条数
    DATA_SIZE = 500
    # 价格分箱边界（美元）
    PRICE_BINS = [0, 25, 50, 100, 250, 500, 1000]
    # 分箱对应的档位标签（与 prompt / 评估一致）
    PRICE_LABELS = ['budget', 'economy', 'mid', 'premium', 'luxury', 'ultra']
    # 随机种子，保证合成数据可复现
    RANDOM_SEED = 42

# 实例化一份全局配置
config = Config()
# 固定 numpy 随机性（生成价格/评分时用）
np.random.seed(config.RANDOM_SEED)
# 固定 torch 随机性（神经网络权重初始化等）
torch.manual_seed(config.RANDOM_SEED)
# 确保 ./data 目录存在，后面存图与 csv
os.makedirs("./data", exist_ok=True)


In [ ]:
# ========== 合成亚马逊风格商品表：标题/描述/价格/类目/品牌/评分 ==========

# 进度提示（emoji 在 print 字符串里，属可运行输出文案，保持原样）
print("📦 Generating synthetic product data...")

# 类目、品牌、各品类标题模板（英文字段，模拟课程数据结构）
CATEGORIES = ["Beauty", "Electronics", "Clothing", "Home", "Sports"]
BRANDS = ["BrandA", "BrandB", "BrandC", "BrandD", "Generic", "Premium Co", "EcoLife", "TechPro"]
TITLE_TEMPLATES = {
    "Beauty": ["Face Cream", "Lipstick Set", "Sunscreen SPF 50", "Hair Serum", "Body Lotion", "Eye Shadow Palette", "Perfume Roll-On", "Cleansing Oil"],
    "Electronics": ["USB-C Cable", "Wireless Earbuds", "Phone Stand", "Power Bank", "Screen Protector", "Smart Watch", "Tablet Case", "Keyboard"],
    "Clothing": ["Cotton T-Shirt", "Denim Jacket", "Running Shorts", "Winter Scarf", "Canvas Sneakers", "Polo Shirt", "Yoga Pants", "Baseball Cap"],
    "Home": ["Desk Lamp", "Throw Pillow", "Storage Bins", "Kitchen Scale", "Coaster Set", "Plant Pot", "Blanket", "Wall Hook"],
    "Sports": ["Resistance Bands", "Water Bottle", "Gym Bag", "Jump Rope", "Yoga Mat", "Dumbbells Set", "Running Belt", "Foam Roller"],
}

# 生成条数与种子（与 Config 对齐）
n = config.DATA_SIZE
np.random.seed(config.RANDOM_SEED)

# 逐条抽样拼 dict，最后进 DataFrame
data = []
for i in range(n):
    # 随机类目
    cat = np.random.choice(CATEGORIES)
    # 标题 = 模板 + 序号后缀，减少完全重复
    title = np.random.choice(TITLE_TEMPLATES[cat]) + f" #{i % 100}"
    # 对数正态采样价格，再夹到 [5, 950]
    price = float(np.clip(np.random.lognormal(3, 1.2), 5, 950))
    # 简单英文描述，带上类目信息
    desc = f"Quality {title.lower()}. Great for daily use. Durable and reliable. Category: {cat}."
    data.append({
        "title": title,
        "description": desc,
        "price": price,
        "category": cat,
        "brand": np.random.choice(BRANDS),
        # 评分近似正态，夹到 1.0–5.0
        "rating": float(np.clip(np.random.normal(4.2, 0.6), 1.0, 5.0)),
    })

# 成表
df = pd.DataFrame(data)
# 按 PRICE_BINS 切成价格档位标签列
df["price_category"] = pd.cut(df["price"], bins=config.PRICE_BINS, labels=config.PRICE_LABELS, right=False)
# 拼文本特征：标题+描述+类目+品牌，供 TF-IDF
df["text"] = df["title"] + " " + df["description"] + " " + df["category"] + " " + df["brand"]

# 摘要与抽样预览
print(f"✅ Generated {len(df)} products | Price: ${df['price'].min():.0f}-${df['price'].max():.0f}")
print(df[["title", "price", "price_category"]].head())


In [ ]:
# ========== 探索性分析：价格/档位/评分分布图，并落盘 csv ==========

# 若还没跑上一格，立刻给出可操作错误（文案保持英文原样）
try:
    _ = len(df)
except NameError:
    raise NameError("'df' not found. Run the previous cell (📦 Generate synthetic data) first, then run all cells in order from the top.")

# 1×3 子图画布
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
# 左：价格直方图
df['price'].hist(bins=20, ax=axes[0], edgecolor='black', color='skyblue')
axes[0].set_title('Price Distribution')
axes[0].set_xlabel('Price ($)')

# 中：各价格档位计数柱状图
df['price_category'].value_counts().sort_index().plot(kind='bar', ax=axes[1], color='lightgreen')
axes[1].set_title('Price Categories')
axes[1].tick_params(axis='x', rotation=45)

# 右：评分直方图
df['rating'].hist(bins=20, ax=axes[2], edgecolor='black', color='salmon')
axes[2].set_title('Rating Distribution')
axes[2].set_xlabel('Rating')

# 紧凑布局，避免标题重叠
plt.tight_layout()
# 确保目录存在后存图
os.makedirs("./data", exist_ok=True)
plt.savefig("./data/exploration.png")
plt.show()
# 原始表存 csv，方便课外复用
df.to_csv("./data/amazon_products.csv", index=False)


In [ ]:
# ========== 准备 JSONL 对话样本：user 描述商品 → assistant 答价格档位 ==========

# 把一行商品格式化成「预测价格档」的英文 prompt（字符串保持原样）
def create_prompt(row):
    return f"""Predict price category for this Amazon product:
Title: {row['title']}
Description: {row['description'][:200]}
Category: {row['category']}
Brand: {row['brand']}
Price category ({'/'.join(config.PRICE_LABELS)}):"""

# 遍历 df，组装 OpenAI 风格 messages 列表
training_data = []
for _, row in df.iterrows():
    training_data.append({
        "messages": [
            {"role": "user", "content": create_prompt(row)},
            # 监督标签：价格档位字符串
            {"role": "assistant", "content": row['price_category']}
        ]
    })

# 写成 jsonl：每行一个 JSON 对象
with open("./data/training_data.jsonl", 'w') as f:
    for ex in training_data:
        f.write(json.dumps(ex) + '\n')
print(f"✅ Prepared {len(training_data)} training examples")


In [ ]:
# ========== 传统 ML 特征：TF-IDF 文本向量 + 标签编码 + 分层切分 ==========

# 最多 500 维 TF-IDF，英文停用词
vectorizer = TfidfVectorizer(max_features=500, stop_words='english')
# 拟合并变换 text 列 → 稀疏矩阵 X_text
X_text = vectorizer.fit_transform(df['text'])

# 把价格档位类别名编成 0..K-1
le = LabelEncoder()
y = le.fit_transform(df['price_category'])

# 80/20 分层切分，保持各类比例
X_train, X_test, y_train, y_test = train_test_split(
    X_text, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train: {X_train.shape[0]}, Test: {X_test.shape[0]}")


In [ ]:
# ========== 训练基线：线性回归 MAE + 若干分类器准确率 ==========

# 结果字典：后面画图与汇总都读它
results = {}

# --- 线性回归：直接预测连续价格，用 MAE 衡量 ---
lr_reg = LinearRegression()
# 真值价格向量
y_price = df['price'].values
# 同一套文本特征上再切一刀（回归任务）
X_tr, X_te, yp_tr, yp_te = train_test_split(X_text, y_price, test_size=0.2, random_state=42)
lr_reg.fit(X_tr, yp_tr)
mae = mean_absolute_error(yp_te, lr_reg.predict(X_te))
results['Linear Regression (MAE)'] = f"${mae:.2f}"
print(f"Linear Regression MAE: ${mae:.2f}")

# --- 分类模型：预测价格档位 ---
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
}
# 若 xgb 导入成功，加入 XGBClassifier
if xgb is not None:
    models['XGBoost'] = xgb.XGBClassifier(n_estimators=100, random_state=42)

# 逐模型 fit → predict → accuracy，写入 results
for name, model in models.items():
    model.fit(X_train, y_train)
    acc = accuracy_score(y_test, model.predict(X_test))
    results[name] = f"{acc:.2%}"
    print(f"{name}: {acc:.2%}")


In [ ]:
# ========== 小型神经网络：把 TF-IDF 稠密化后用 MLP 做档位分类 ==========

# Dataset：把特征/标签转成 Float/Long Tensor
class PriceDataset(Dataset):
    def __init__(self, X, y): 
        # 若是稀疏矩阵则 toarray；已是 ndarray 则直接包
        self.X = torch.FloatTensor(X.toarray()) if hasattr(X, 'toarray') else torch.FloatTensor(X)
        self.y = torch.LongTensor(y)
    def __len__(self): return len(self.y)
    def __getitem__(self, idx): return self.X[idx], self.y[idx]

# 两层隐藏层的简单前馈网络
class SimpleNN(nn.Module):
    def __init__(self, input_dim, num_classes):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 128), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(128, 64), nn.ReLU(),
            nn.Linear(64, num_classes)
        )
    def forward(self, x): return self.net(x)

# 稀疏 → 稠密，供 Linear 层使用
X_train_dense = X_train.toarray()
X_test_dense = X_test.toarray()

# DataLoader：batch=32；训练打乱，测试不打乱（默认）
train_dataset = PriceDataset(X_train_dense, y_train)
test_dataset = PriceDataset(X_test_dense, y_test)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32)

# 输入维 = TF-IDF 维；输出类数 = 价格档位数
model = SimpleNN(X_train_dense.shape[1], len(le.classes_))
# 多分类交叉熵
criterion = nn.CrossEntropyLoss()
# Adam 优化器，学习率 1e-3
optimizer = optim.Adam(model.parameters(), lr=0.001)

# --- 训练 20 个 epoch，记录平均 loss ---
train_losses = []
for epoch in range(20):
    model.train()
    epoch_loss = 0
    for Xb, yb in train_loader:
        optimizer.zero_grad()
        loss = criterion(model(Xb), yb)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    train_losses.append(epoch_loss/len(train_loader))
    if (epoch+1) % 5 == 0:
        print(f"Epoch {epoch+1}/20, Loss: {epoch_loss/len(train_loader):.4f}")

# --- 评估准确率 ---
model.eval()
correct = 0
total = 0
with torch.no_grad():
    for Xb, yb in test_loader:
        outputs = model(Xb)
        _, predicted = torch.max(outputs, 1)
        total += yb.size(0)
        correct += (predicted == yb).sum().item()
nn_acc = correct / total
results['Neural Network'] = f"{nn_acc:.2%}"
print(f"Neural Network Accuracy: {nn_acc:.2%}")

# 画训练损失曲线并保存
plt.figure(figsize=(8, 4))
plt.plot(train_losses)
plt.title('Neural Network Training Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.savefig("./data/nn_loss.png")
plt.show()


In [ ]:
# ==========（可选）OpenRouter 零样本：抽 20 条测几个云端模型 ==========

# 只有配置了 Key 才跑，避免无密钥时报错
if config.OPENROUTER_API_KEY:
    # OpenAI 兼容客户端，指向 OpenRouter
    client = OpenAI(api_key=config.OPENROUTER_API_KEY, base_url=OPENROUTER_BASE_URL)

    # 调 chat.completions；temperature=0、短 max_tokens，只要档位词
    def test_model(model_name, prompt):
        try:
            resp = client.chat.completions.create(
                model=model_name,
                messages=[{"role": "user", "content": prompt}],
                temperature=0, max_tokens=10
            )
            return resp.choices[0].message.content.strip().lower()
        except Exception as e:
            print(f"Error: {e}")
            return None

    # 固定种子抽样，最多 20 条
    test_df = df.sample(min(20, len(df)), random_state=42)
    # 要对比的模型 id 列表（保持原样）
    models_to_test = [
        "openai/gpt-4o-mini", 
        "openai/gpt-3.5-turbo",
        "anthropic/claude-3-haiku"
    ]

    # 逐模型统计 exact match 准确率
    for model in models_to_test:
        correct = 0
        total = 0
        for _, row in test_df.iterrows():
            pred = test_model(model, create_prompt(row))
            if pred:
                total += 1
                if pred == row['price_category']:
                    correct += 1
        if total > 0:
            acc = correct/total
            model_name = model.split('/')[-1]
            results[f"{model_name} (0-shot)"] = f"{acc:.2%}"
            print(f"{model_name}: {acc:.2%} ({correct}/{total})")


In [ ]:
# ========== 汇总对比：分类准确率条形图 + 打印全部 results ==========

# 含 '%' 的条目当分类结果；回归 MAE 单独取出
class_results = {k: v for k, v in results.items() if '%' in v}
reg_result = results.get('Linear Regression (MAE)', 'N/A')

# 有分类结果才画横条图
if class_results:
    plt.figure(figsize=(10, 5))
    models = list(class_results.keys())
    # '12.34%' → 0.1234
    accs = [float(v.strip('%'))/100 for v in class_results.values()]
    # 零样本 LLM 用一种颜色，传统模型用另一种
    colors = ['#FF9999' if '0-shot' in m else '#66B2FF' for m in models]
    bars = plt.barh(models, accs, color=colors)
    plt.xlabel('Accuracy')
    plt.title(f'Model Comparison (Regression MAE: {reg_result})')
    # 在条末端标准确率
    for bar, acc in zip(bars, accs):
        plt.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2, 
                f'{acc:.1%}', va='center')
    plt.tight_layout()
    plt.savefig("./data/model_comparison.png")
    plt.show()

# 文本总表
print("\n📊 FINAL RESULTS:")
print("-" * 40)
for model, result in results.items():
    print(f"{model:30} {result}")


In [ ]:
# ========== Gradio Demo：输入标题/描述/类目/品牌 → 调 LLM 预测档位 ==========

class PricePredictorApp:
    def __init__(self):
        # 有 Key 才建客户端，否则 predict 返回配置提示
        self.client = OpenAI(api_key=config.OPENROUTER_API_KEY, base_url=OPENROUTER_BASE_URL) if config.OPENROUTER_API_KEY else None

    # Gradio 回调：四段输入 → Markdown 结果
    def predict(self, title, desc, category, brand):
        if not self.client: 
            return "❌ OpenRouter API key not configured. Add to .env file."

        # 与训练/评测同风格的英文 prompt（保持原样）
        prompt = f"""Predict price category for this Amazon product:
Title: {title}
Description: {desc}
Category: {category}
Brand: {brand}
Price category ({'/'.join(config.PRICE_LABELS)}):"""

        try:
            resp = self.client.chat.completions.create(
                model=config.BASE_MODEL,
                messages=[{"role": "user", "content": prompt}],
                temperature=0, max_tokens=10
            )
            pred = resp.choices[0].message.content.strip().lower()

            # 档位 → 人类可读价格区间说明
            ranges = {
                'budget': '$0-25', 'economy': '$25-50', 'mid': '$50-100',
                'premium': '$100-250', 'luxury': '$250-500', 'ultra': '$500-1000'
            }
            price_range = ranges.get(pred, 'Unknown')

            return f"""### ✅ Prediction Result
**Price Category:** {pred.upper()}
**Estimated Price:** {price_range}
**Confidence:** High (based on product attributes)"""

        except Exception as e:
            return f"❌ Error: {str(e)}"

# 用 df 第一行当 Gradio examples（若有数据）
sample = df.iloc[0] if len(df) > 0 else None
examples = [[sample['title'], sample['description'][:100], sample['category'], sample['brand']]] if sample is not None else []

# 仅当有 API Key 时构建界面（默认不 launch，避免占端口）
if config.OPENROUTER_API_KEY:
    app = PricePredictorApp()
    iface = gr.Interface(
        fn=app.predict,
        inputs=[
            gr.Textbox(label="📦 Product Title", placeholder="Enter product title..."),
            gr.Textbox(label="📝 Description", lines=3, placeholder="Enter product description..."),
            gr.Dropdown(choices=df['category'].unique().tolist() if len(df) > 0 else ['Beauty'], 
                       label="🏷️ Category", value='Beauty'),
            gr.Textbox(label="🏢 Brand", placeholder="Enter brand name...")
        ],
        outputs=gr.Markdown(label="🎯 Result"),
        title="🛍️ Amazon Price Predictor",
        description="Predict price category from product description using AI",
        examples=examples if examples else None,
        theme="soft"
    )
    # iface.launch(share=True) # 取消注释启动
    print("✅ Gradio app created. Run 'iface.launch()' to start")


In [ ]:
# ==========（可选）把合成数据集推到 Hugging Face Hub ==========

def upload_to_hub():
    """Upload dataset to Hugging Face Hub"""
    try:
        # pandas → HF Dataset
        hf_dataset = HFDataset.from_pandas(df)

        # 80/20 切分
        train_test = hf_dataset.train_test_split(test_size=0.2, seed=42)

        # 交互输入用户名，拼数据集路径
        username = input("Enter your Hugging Face username: ")
        dataset_name = f"{username}/amazon-price-predictor"
        # 推送（需已 huggingface-cli login）
        train_test.push_to_hub(dataset_name)
        print(f"✅ Dataset uploaded to https://huggingface.co/datasets/{dataset_name}")
    except Exception as e:
        print(f"Error uploading: {e}")

# 取消注释以下函数调用可上传：upload_to_hub()
# 上传到集线器（）


In [ ]:
# ========== 快速抽查：对两个手工样例调用 predict_price（需你已定义该函数） ==========

def quick_test():
    """Test with sample products"""
    # 两个风格差异大的商品：高价电子 vs 基础服饰
    test_cases = [
        {
            "title": "Apple MacBook Pro 16-inch",
            "desc": "Powerful laptop with M3 chip, 36GB RAM, 1TB SSD",
            "category": "Electronics",
            "brand": "Apple"
        },
        {
            "title": "Basic Cotton T-Shirt",
            "desc": "Comfortable 100% cotton t-shirt, machine washable",
            "category": "Clothing",
            "brand": "Hanes"
        }
    ]

    print("\n🔍 QUICK TEST RESULTS:")
    print("="*50)
    for tc in test_cases:
        print(f"\nTesting: {tc['title']}")
        # 注意：原笔记本调用 predict_price；若未自行定义，运行会 NameError
        print(predict_price(tc['title'], tc['desc'], tc['category'], tc['brand']))

# 取消注释可运行：quick_test()
# 快速测试（）


In [ ]:
# ========== 落盘：结果 JSON +（可选）joblib 保存向量器与分类模型 ==========

# 时间戳，避免覆盖旧结果文件
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
# 结构化摘要，便于复盘对比
results_summary = {
    "timestamp": timestamp,
    "dataset_size": len(df),
    "price_range": [float(df['price'].min()), float(df['price'].max())],
    "model_results": results,
    "categories": df['category'].unique().tolist()
}

# 写入 ./data/results_*.json
with open(f"./data/results_{timestamp}.json", 'w') as f:
    json.dump(results_summary, f, indent=2)

# 尝试用 joblib 持久化 TF-IDF / LabelEncoder / 各分类器
joblib_dict = {}
try:
    import joblib
    joblib.dump(vectorizer, "./data/vectorizer.pkl")
    joblib.dump(le, "./data/label_encoder.pkl")
    for name, model in models.items():
        joblib.dump(model, f"./data/{name.replace(' ', '_').lower()}.pkl")
    print("✅ Models saved to ./data/")
except:
    print("⚠️ Install joblib to save models: pip install joblib")

print(f"\n✅ Results saved to ./data/results_{timestamp}.json")
print(f"\n📊 Final Summary:")
print(f"Dataset: {len(df)} products")
print(f"Price range: ${df['price'].min():.2f} - ${df['price'].max():.2f}")
# 从分类结果里挑准确率最高的名字
if class_results:
    best = max(class_results.items(), key=lambda x: float(x[1].strip('%')))[0]
    print(f"Best model: {best}")


In [ ]:
# ========== 一键打印流水线摘要：数据概况 + 各模型成绩 ==========

def run_pipeline():
    """Run complete pipeline"""
    print("="*60)
    print("🏷️ AMAZON PRICE PREDICTOR - WEEK 6 PROJECT")
    print("="*60)

    # 1. 数据：前面单元格已生成 df
    print("\n1️⃣ LOADING DATA")
    print("-"*40)
    # 数据已加载到先前的单元格中

    # 2. 打印规模与价格范围
    print("\n2️⃣ DATA SUMMARY")
    print("-"*40)
    print(f"Total products: {len(df)}")
    print(f"Categories: {df['category'].nunique()}")
    print(f"Brands: {df['brand'].nunique()}")
    print(f"Price range: ${df['price'].min():.2f} - ${df['price'].max():.2f}")

    # 3. 打印 results 字典
    print("\n3️⃣ MODEL PERFORMANCE")
    print("-"*40)
    for model, result in results.items():
        print(f"{model:30} {result}")

    print("\n" + "="*60)
    print("✅ PIPELINE COMPLETE!")
    print("="*60)

    return df

# 执行摘要并仍把 df 绑定到返回值（逻辑保持原样）
df = run_pipeline()

# 下一步提示（英文可运行/展示文案保持）
print("\n📝 Next steps:")
print("1. Run 'quick_test()' to test predictions")
print("2. Launch Gradio app with 'iface.launch()'")
print("3. Upload to Hugging Face with 'upload_to_hub()'")
